# Social Media Sentiment (TweetEval)

This started as a Facebook post sentiment project, but I couldn't find a public Facebook post dataset with trustworthy labels. The closest real, openly available thing is TweetEval's sentiment task (Barbieri et al., 2020, from SemEval-2017 Task 4): English tweets hand-labeled as negative, neutral, or positive. I load it from the `cardiffnlp/tweet_eval` repo on Hugging Face and use its official splits (45,615 train / 12,284 test). Same shape as a real social-listening setup.

Three-class problem here (pos/neg/neutral), which is a step up from the binary case.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

np.random.seed(11)
DATA_DIR = Path("../../data/tweet_eval_sentiment")
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE = "https://huggingface.co/datasets/cardiffnlp/tweet_eval/resolve/main/sentiment/"
NAMES = {0: "negative", 1: "neutral", 2: "positive"}
splits = {}
for split in ["train", "test"]:
    path = DATA_DIR / f"{split}.parquet"
    if not path.exists():
        pd.read_parquet(BASE + f"{split}-00000-of-00001.parquet").to_parquet(path)
    d = pd.read_parquet(path)
    d["sentiment"] = d["label"].map(NAMES)
    splits[split] = d
train_df, test_df = splits["train"], splits["test"]
print(train_df.shape, test_df.shape)
train_df.sentiment.value_counts()

(45615, 3) (12284, 3)


sentiment
neutral     20673
positive    17849
negative     7093
Name: count, dtype: int64

In [2]:
X_train, y_train = train_df["text"].values, train_df["sentiment"].values
X_test, y_test = test_df["text"].values, test_df["sentiment"].values
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=2000, C=2.0)
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)

print(classification_report(y_test, pred, digits=3))
# TweetEval's official sentiment metric is macro recall
from sklearn.metrics import recall_score
print(f"macro recall (TweetEval metric): {recall_score(y_test, pred, average='macro'):.3f}")
print("confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=["positive","neutral","negative"]),
                   index=["pos","neu","neg"], columns=["pos","neu","neg"]))

              precision    recall  f1-score   support

    negative      0.666     0.394     0.495      3972
     neutral      0.598     0.720     0.653      5937
    positive      0.523     0.612     0.564      2375

    accuracy                          0.594     12284
   macro avg      0.596     0.576     0.571     12284
weighted avg      0.605     0.594     0.585     12284

macro recall (TweetEval metric): 0.576
confusion matrix (rows=true, cols=pred):
      pos   neu   neg
pos  1454   833    88
neu   963  4277   697
neg   363  2043  1566


## Takeaway

On the official 12,284-tweet test split, TF-IDF + logistic regression gets 59.4% accuracy and 0.576 macro recall (TweetEval's headline metric for this task). For comparison, the TweetEval paper's RoBERTa baselines score in the low 70s on the same metric, so a bag-of-words model leaves a lot on the table here.

The interesting part is where it fails. Negative tweets are the problem: recall is only 0.39, and more than half of the true negatives (2,043 of 3,972) get called neutral. The test set is also skewed differently from train (32% negative in test vs 16% in train), which makes the negative class look worse than it would on a split with the same mix. Neutral is the catch-all. It soaks up mistakes from both sides, because "neutral" can mean anything from a factual statement to sarcasm the model can't read.

If I were doing this for real I'd:

1. Use a transformer model trained on Twitter/social text (cardiffnlp/twitter-roberta-base-sentiment).
2. Treat the neutral class with care, maybe a two-step pipeline (subjectivity classifier, then polarity classifier).
3. Calibrate the probabilities, because downstream consumers (alerting dashboards, etc) want a confidence score.